In [ ]:
! pip install torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu118
! pip install transformers==4.46.3 torch tokenizers==0.20.3 PyMuPDF img2pdf einops easydict addict Pillow numpy
! pip install peft trl datasets accelerate bitsandbytes

In [ ]:
! pip install flash-attn==2.7.3 --no-build-isolation -v

In [ ]:
! pip install transformers==4.47.1

In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer

model_name = 'deepseek-ai/DeepSeek-OCR-2'
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModel.from_pretrained(
        model_name,
        _attn_implementation='eager',
        torch_dtype=torch.bfloat16,
        device_map="cuda",
        trust_remote_code=True,
        use_safetensors=True,
        force_download=True
    )

In [ ]:
from datasets import load_dataset
dataset = load_dataset("blue7012/UIT_HWDB")

In [ ]:
instruction = "<image>\nFree OCR. "

def convert_to_conversation(sample):
    """Convert dataset sample to conversation format"""
    conversation = [
        {
            "role": "<|User|>",
            "content": instruction,
            "images": [sample['image']]
        },
        {
            "role": "<|Assistant|>",
            "content": sample["text"]
        },
    ]
    return {"messages": conversation}


In [ ]:
converted_train_dataset = [convert_to_conversation(sample) for sample in dataset["train"]]

In [ ]:
converted_test_dataset = [convert_to_conversation(sample) for sample in dataset["test"]]

In [ ]:
print(model)

In [ ]:
target_modules = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj",
],
r = 8,
lora_alpha = 8,
lora_dropout = 0.1,
bias = "none",
random_state = 3407,
use_rslora = False,
loftq_config = None,
task_type = "QUESTION_ANS"

In [ ]:
from peft import LoraConfig, get_peft_model

peft_config = LoraConfig(
    r=8,
    lora_alpha=8,
    lora_dropout=0.1,
    bias="none",
    target_modules=".*(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj).*",
)

In [ ]:
peft_model = get_peft_model(model, peft_config)
peft_model.print_trainable_parameters()

In [ ]:
from huggingface_hub import snapshot_download
import os

snapshot_download(
    repo_id="deepseek-ai/DeepSeek-OCR-2",
    local_dir="deepseek_ocr2",
    allow_patterns=["*.py"]
)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
data_collator = DeepSeekOCR2DataCollator(
    tokenizer = tokenizer,
    model = peft_model,
    image_size = 768,
    base_size = 1024,
    crop_mode = True,
    train_on_responses_only = True,
)

In [ ]:
from transformers import TrainingArguments, Trainer
import os

drive_output_dir = "/content/drive/MyDrive/deepseek_ocr_training"
os.makedirs(drive_output_dir, exist_ok=True)

training_args = TrainingArguments(
    output_dir=drive_output_dir,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=3,
    logging_steps=10,
    save_steps=100,
    evaluation_strategy="steps",
    eval_steps=200,
    save_total_limit=3,
    bf16=True,
    optim="adamw_torch",
    remove_unused_columns=False,
    report_to="none"
)

In [ ]:
trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=converted_train_dataset,
    eval_dataset=converted_test_dataset,
    data_collator=data_collator,
    tokenizer=tokenizer,
)
trainer.train()

In [ ]:
peft_model.save_pretrained("./deepseek-ocr-lora-adapter")
tokenizer.save_pretrained("./deepseek-ocr-lora-adapter")
peft_model.save_pretrained("/content/drive/MyDrive/deepseek-ocr-lora-adapter")
tokenizer.save_pretrained("/cotent/drive/MyDrive/deepseek-ocr-lora-adapter")
peft_model.push_to_hub("blue7012/deepseek_ocr_lora", token = "hf_xxx")
tokenizer.push_to_hub("blue7012/deepseek_ocr_lora", token = "hf_xxx")

In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModel, AutoTokenizer

adapter_path = "./deepseek-ocr-lora-adapter"

model_name = 'deepseek-ai/DeepSeek-OCR-2'
model = AutoModel.from_pretrained(
        model_name,
        _attn_implementation='eager',
        torch_dtype=torch.bfloat16,
        device_map="cuda",
        trust_remote_code=True,
        use_safetensors=True
    )
tokenizer = AutoTokenizer.from_pretrained("./deepseek-ocr-lora-adapter", trust_remote_code=True)
model = PeftModel.from_pretrained(model, adapter_path)

model.eval()